# Planning

**Planning** finds a sequence of actions that reaches a goal, like [search](search.ipynb), but the states are not nodes of a graph written out in advance. They are **described by facts**, and the actions by what they require and what they change. The planner works out by itself which actions apply in each state and where they lead, so the same program plans in any domain described this way.

The classic formulation is **STRIPS** (Richard Fikes and Nils Nilsson, 1971), written for Shakey, the first mobile robot that planned its own actions. Its descendant, **PDDL** (1998), is the standard language of today's planners.

## States and actions as facts

The **blocks world**: blocks on a table, stacked on each other, and one action, moving a block that has nothing on top of it onto the table or onto another clear block. A **state** is a set of facts such as `("on", "C", "A")` (C is on A) and `("clear", "B")` (nothing is on B). An **action** has three sets of facts:

- **preconditions**, which must hold for the action to apply;
- an **add list**, the facts it makes true;
- a **delete list**, the facts it makes false.

In [ ]:
from itertools import product

blocks = ["A", "B", "C"]
places = blocks + ["table"]

actions = []
for block, origin, destination in product(blocks, places, places):
    if len({block, origin, destination}) < 3:
        continue
    destination_must_be_clear = {("clear", destination)} if destination != "table" else set()
    origin_becomes_clear = {("clear", origin)} if origin != "table" else set()
    actions.append({
        "name": f"move {block} from {origin} to {destination}",
        "preconditions": {("on", block, origin), ("clear", block)} | destination_must_be_clear,
        "add": {("on", block, destination)} | origin_becomes_clear,
        "delete": {("on", block, origin)} | destination_must_be_clear,
    })

print(len(actions), "actions, for example:", actions[0]["name"])
for part in ["preconditions", "add", "delete"]:
    print(f"  {part:13}", sorted(actions[0][part]))

18 actions, for example: move A from B to C
  preconditions [('clear', 'A'), ('clear', 'C'), ('on', 'A', 'B')]
  add           [('clear', 'B'), ('on', 'A', 'C')]
  delete        [('clear', 'C'), ('on', 'A', 'B')]


The table is always free, so moving a block onto it needs no clear-destination precondition. Applying an action to a state is set arithmetic: remove the delete list, then add the add list.

In [ ]:
def applicable(action, state):
    return action["preconditions"] <= state


def apply(action, state):
    return frozenset((state - action["delete"]) | action["add"])

## The Sussman anomaly

The start: C on A, with A and B on the table. The goal: A on B and B on C, a tower of A, B and C from top to bottom.

Gerald Sussman pointed out in the early 1970s that this simple problem defeats a planner that tackles the goals one at a time. Putting A on B first leaves C still on A, and A must be moved off again; putting B on C first buries A under the tower. Solving it needs the goals interleaved.

In [ ]:
start = frozenset({("on", "C", "A"), ("on", "A", "table"), ("on", "B", "table"), ("clear", "C"), ("clear", "B")})
goal = {("on", "A", "B"), ("on", "B", "C")}

Planning as breadth-first search over the states: the frontier holds states with the plan that reached them, and a state is a goal state when it contains every goal fact.

In [ ]:
from collections import deque


def plan_by_breadth_first_search(start, goal, actions):
    frontier = deque([(start, [])])
    visited = {start}
    while frontier:
        state, plan = frontier.popleft()
        if goal <= state:
            return plan, len(visited)
        for action in actions:
            if applicable(action, state):
                next_state = apply(action, state)
                if next_state not in visited:
                    visited.add(next_state)
                    frontier.append((next_state, plan + [action["name"]]))
    return None, len(visited)


plan, states_seen = plan_by_breadth_first_search(start, goal, actions)
for step, action_name in enumerate(plan, 1):
    print(step, action_name)
print("States seen:", states_seen)

1 move C from A to table
2 move B from table to C
3 move A from table to B
States seen: 13


The search found the shortest plan, three moves, after seeing 13 states: with three blocks there are exactly 13 ways to arrange them, so it saw them all.

## Planning at scale

Blind search doesn't scale: the number of arrangements grows explosively with the number of blocks. Real planners use **heuristics** computed from the action descriptions themselves. The FF planner (Jörg Hoffmann and Bernhard Nebel, 2001), for example, estimates the distance to the goal by solving a relaxed problem that ignores every delete list, a problem easy enough to solve at every step. Planning runs logistics and manufacturing schedules, and it flew in space: NASA's Remote Agent planned and executed the operations of the Deep Space 1 probe, autonomously, in 1999. Video games use it too: the enemies of *F.E.A.R.* (2005) chose their tactics with a STRIPS-style planner.

Agents built on language models face the same problem, choosing a sequence of tool calls that reaches a goal, but they describe the actions in natural language instead of preconditions and effects, so they gain flexibility and lose the planner's guarantees.